In [28]:
%load_ext autoreload
%autoreload 2
import json

from backend.app.vendor import GeminiAgent
from backend.app import prompts

import typing_extensions as typing

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [41]:
class StoryNotes:

    def __init__(self):
        self.characters = ""
        self.setting = ""
        self.timeline = ""
        self.themes = ""

    def update_characters(self, characters: str):
        self.characters = characters
        return f"New characters list: {characters}"
    
    def update_setting(self, setting: str):
        self.setting = setting
        return f"New setting: {setting}"
    
    def update_timeline(self, timeline: str):
        self.timeline = timeline
        return f"New timeline: {timeline}"
    
    def update_themes(self, themes: str):
        self.themes = themes
        return f"New themes: {themes}"

    def get_characters(self):
        return self.characters
    
    def get_setting(self):
        return self.setting
    
    def get_timeline(self):
        return self.timeline
    
    def get_themes(self):
        return self.themes
    
    def get_current_context(self):
        return f"""
Characters
This section includes attributes of characters such as their personality and character development
{self.characters}

Setting
This section includes the setting of the story, including descriptive details
{self.setting}

Timeline
This section includes the timeline of the story, including important dates and events
{self.timeline}

Themes
This section includes the themes of the story, including the main ideas and messages
{self.themes}
"""

class Tool:
    def __init__(self, name: str, description: str, state: StoryNotes):
        self.name = name
        self.description = description
        self.state = state
    
    def __call__(self, input: str):
        pass


class UpdateCharacters(Tool):
    name = "update_characters"
    description = "Update the characters of the story"

    def __init__(self, state: StoryNotes):
        super().__init__(self.name, self.description, state)

    def __call__(self, input: str):
        self.state.update_characters(input)

class UpdateSetting(Tool):
    name = "update_setting"
    description = "Update the setting of the story"

    def __init__(self, state: StoryNotes):
        super().__init__(self.name, self.description, state)

    def __call__(self, input: str):
        self.state.update_setting(input)

class UpdateTimeline(Tool):
    name = "update_timeline"
    description = "Update the timeline of the story"

    def __init__(self, state: StoryNotes):
        super().__init__(self.name, self.description, state)

    def __call__(self, input: str):
        self.state.update_timeline(input)

class UpdateThemes(Tool):
    name = "update_themes"
    description = "Update the themes of the story"

    def __init__(self, state: StoryNotes):
        super().__init__(self.name, self.description, state)

    def __call__(self, input: str):
        self.state.update_themes(input)

class FunctionCall(typing.TypedDict):
    function_name: str
    function_input: str

class Analysis(typing.TypedDict):
    reply: str
    functions: list[FunctionCall]

class ConversationalAgent(GeminiAgent):
    general_description: str
    prompt_template: str = prompts.CHAT_PROMPT
    state = StoryNotes()
    tools= {
        "update_characters": UpdateCharacters(state),
        "update_setting": UpdateSetting(state),
        "update_timeline": UpdateTimeline(state),
        "update_themes": UpdateThemes(state)
    }

    def __init__(self, model_name: str, system_prompt: str):
        super().__init__(model_name=model_name, system_prompt=system_prompt)

    def run(self, query, conversation_history: list[dict]):
        characters = self.state.get_characters()
        setting = self.state.get_setting()
        timeline = self.state.get_timeline()
        themes = self.state.get_themes()
        current_context = self.state.get_current_context()

        prompt = self.prompt_template.format(message=query, current_context=current_context)
        generated = self.reply_structured(prompt, conversation_history, Analysis)
        result = json.loads(generated.candidates[0].content.parts[0].text)
        for tool in result["functions"]:
            self.tools[tool["function_name"]](tool["function_input"])
        else:
            return result["reply"]
            
if __name__ == "__main__":
    agent = ConversationalAgent(model_name="gemini-1.5-flash", system_prompt="""You are a ghostwriter gathering notes for a memoir about the subject.""")
    print(agent.state.get_current_context())
    print("-"*100)
    reply = agent.run("My name is Henry and I like popsicles", [])
    print(agent.state.get_current_context())
    print("-"*100)
    reply = agent.run("My sister's name is Caroline and she likes dogs", [])
    print(agent.state.get_current_context())
    print("-"*100)
    print(reply)


Characters
This section includes attributes of characters such as their personality and character development


Setting
This section includes the setting of the story, including descriptive details


Timeline
This section includes the timeline of the story, including important dates and events


Themes
This section includes the themes of the story, including the main ideas and messages


----------------------------------------------------------------------------------------------------

Characters
This section includes attributes of characters such as their personality and character development
Henry, favorite food is popsicles

Setting
This section includes the setting of the story, including descriptive details


Timeline
This section includes the timeline of the story, including important dates and events


Themes
This section includes the themes of the story, including the main ideas and messages


----------------------------------------------------------------------------------